In [9]:
import os
import time

import librosa
import numpy as np
import pandas as pd

from xgboost import XGBClassifier

from sklearn.model_selection import (
    train_test_split
)

from sklearn.preprocessing import (
    LabelEncoder,
    StandardScaler
)

from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support
)

from sklearn.neighbors import (
    KNeighborsClassifier
)

from sklearn.svm import SVC

from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    AdaBoostClassifier
)

from sklearn.tree import (
    DecisionTreeClassifier
)

from sklearn.linear_model import (
    LogisticRegression
)

from sklearn.naive_bayes import (
    GaussianNB
)

from sklearn.neural_network import (
    MLPClassifier
)

In [13]:
DATASET = "/media/feliciano/Aux/AI_AFS_DATASET/behavior_dataset"

CLASSES = [
    "normal",
    "clustering",
    "agitation",
    "Background"
]

SR = 16000

In [14]:
X = []
y = []

for label in CLASSES:

    folder = os.path.join(
        DATASET,
        label
    )

    files = [

        f for f in os.listdir(folder)

        if f.endswith(".wav")
    ]

    print(
        f"{label}: {len(files)} files"
    )

    for file in files:

        path = os.path.join(
            folder,
            file
        )

        try:

            signal, sr = librosa.load(
                path,
                sr=SR,
                mono=True
            )

            signal = (
                signal - np.mean(signal)
            ) / (
                np.std(signal) + 1e-8
            )

            features = []

            # RMS

            features.append(
                np.mean(
                    librosa.feature.rms(
                        y=signal
                    )
                )
            )

            # ZCR

            features.append(
                np.mean(
                    librosa.feature.zero_crossing_rate(
                        signal
                    )
                )
            )

            # Spectral

            features.append(
                np.mean(
                    librosa.feature.spectral_centroid(
                        y=signal,
                        sr=sr
                    )
                )
            )

            features.append(
                np.mean(
                    librosa.feature.spectral_bandwidth(
                        y=signal,
                        sr=sr
                    )
                )
            )

            features.append(
                np.mean(
                    librosa.feature.spectral_rolloff(
                        y=signal,
                        sr=sr
                    )
                )
            )

            # Contrast

            contrast = librosa.feature.spectral_contrast(
                y=signal,
                sr=sr
            )

            features.extend(
                np.mean(
                    contrast,
                    axis=1
                )
            )

            # Log Mel

            mel = librosa.feature.melspectrogram(
                y=signal,
                sr=sr,
                n_mels=64
            )

            mel = librosa.power_to_db(
                mel,
                ref=np.max
            )

            features.extend(
                np.mean(
                    mel,
                    axis=1
                )
            )

            X.append(features)
            y.append(label)

        except Exception as e:

            print(
                path,
                e
            )

normal: 8049 files
clustering: 1260 files
agitation: 1140 files
Background: 1500 files


In [15]:
X = np.array(
    X,
    dtype=np.float32
)

y = np.array(y)

print("\nSamples:", len(X))
print("Features:", X.shape[1])

encoder = LabelEncoder()

y_encoded = encoder.fit_transform(y)

for i, c in enumerate(
    encoder.classes_
):
    print(i, c)


Samples: 11949
Features: 76
0 Background
1 agitation
2 clustering
3 normal


In [16]:
X_train, X_test, y_train, y_test = train_test_split(

    X,
    y_encoded,

    test_size=0.20,

    stratify=y_encoded,

    random_state=42
)

In [17]:
models = {

    "Logistic Regression": Pipeline([

        (
            "scaler",
            StandardScaler()
        ),

        (
            "model",
            LogisticRegression(

                C=10,

                max_iter=3000,

                class_weight="balanced",

                random_state=42
            )
        )
    ]),

    "Gaussian NB": GaussianNB(

        var_smoothing=1e-8
    ),

    "Decision Tree": DecisionTreeClassifier(

        criterion="gini",

        max_depth=20,

        min_samples_leaf=2,

        min_samples_split=5,

        class_weight="balanced",

        random_state=42
    ),

    "Random Forest": RandomForestClassifier(

        n_estimators=1000,

        max_depth=None,

        min_samples_split=4,

        min_samples_leaf=2,

        max_features="sqrt",

        class_weight="balanced_subsample",

        random_state=42,

        n_jobs=-1
    ),

    "Extra Trees": ExtraTreesClassifier(

        n_estimators=1000,

        max_features="sqrt",

        min_samples_leaf=2,

        min_samples_split=4,

        class_weight="balanced_subsample",

        random_state=42,

        n_jobs=-1
    ),

    "AdaBoost": AdaBoostClassifier(

        n_estimators=1000,

        learning_rate=0.05,

        random_state=42
    ),

    "XGBoost": XGBClassifier(

        n_estimators=1000,

        max_depth=8,

        learning_rate=0.03,

        subsample=0.9,

        colsample_bytree=0.9,

        gamma=0.1,

        min_child_weight=3,

        reg_alpha=0.1,

        reg_lambda=2,

        objective="multi:softprob",

        num_class=len(CLASSES),

        eval_metric="mlogloss",

        random_state=42,

        n_jobs=-1
    ),

    "KNN": Pipeline([

        (
            "scaler",
            StandardScaler()
        ),

        (
            "model",
            KNeighborsClassifier(

                n_neighbors=9,

                weights="distance",

                metric="minkowski",

                p=2
            )
        )
    ]),

    "SVM (RBF)": Pipeline([

        (
            "scaler",
            StandardScaler()
        ),

        (
            "model",
            SVC(

                kernel="rbf",

                C=50,

                gamma=0.005,

                class_weight="balanced",

                random_state=42
            )
        )
    ]),

    "MLP": Pipeline([

        (
            "scaler",
            StandardScaler()
        ),

        (
            "model",
            MLPClassifier(

                hidden_layer_sizes=(
                    256,
                    128,
                    64
                ),

                activation="relu",

                solver="adam",

                alpha=0.0001,

                learning_rate="adaptive",

                learning_rate_init=0.001,

                batch_size=64,

                early_stopping=True,

                validation_fraction=0.1,

                n_iter_no_change=15,

                max_iter=1000,

                random_state=42
            )
        )
    ])
}

In [19]:
results = []

for name, model in models.items():

    print("\n")
    print("=" * 80)
    print(name)
    print("=" * 80)

    start = time.time()

    model.fit(
        X_train,
        y_train
    )

    training_time = (
        time.time() - start
    )

    pred = model.predict(
        X_test
    )

    acc = accuracy_score(
        y_test,
        pred
    )

    precision, recall, f1, _ = \
        precision_recall_fscore_support(
            y_test,
            pred,
            average="macro"
        )

    results.append({

        "Model": name,

        "Accuracy": round(acc, 4),

        "Precision": round(precision, 4),

        "Recall": round(recall, 4),

        "F1": round(f1, 4),

        "Training_Time_s":
        round(training_time, 2)
    })

    print(
        f"\nAccuracy : {acc:.4f}"
    )

    print(
        f"Precision: {precision:.4f}"
    )

    print(
        f"Recall   : {recall:.4f}"
    )

    print(
        f"F1 Score : {f1:.4f}"
    )

    print(
        f"Training : {training_time:.2f}s"
    )

    print(
        classification_report(
            y_test,
            pred,
            target_names=encoder.classes_
        )
    )

    print(
        confusion_matrix(
            y_test,
            pred
        )
    )



Logistic Regression

Accuracy : 0.8389
Precision: 0.7815
Recall   : 0.8944
F1 Score : 0.8160
Training : 2.70s
              precision    recall  f1-score   support

  Background       1.00      1.00      1.00       300
   agitation       0.43      0.83      0.57       228
  clustering       0.73      0.95      0.83       252
      normal       0.97      0.79      0.87      1610

    accuracy                           0.84      2390
   macro avg       0.78      0.89      0.82      2390
weighted avg       0.89      0.84      0.85      2390

[[ 300    0    0    0]
 [   0  190    4   34]
 [   0    2  240   10]
 [   0  250   85 1275]]


Gaussian NB

Accuracy : 0.7075
Precision: 0.6725
Recall   : 0.8073
F1 Score : 0.6938
Training : 0.01s
              precision    recall  f1-score   support

  Background       1.00      0.99      0.99       300
   agitation       0.32      0.64      0.43       228
  clustering       0.44      0.97      0.61       252
      normal       0.93      0.62      

In [21]:
results_df = pd.DataFrame(
    results
)

results_df = results_df.sort_values(

    by="F1",

    ascending=False
)

print("\n")
print("=" * 80)
print("FINAL MODEL COMPARISON")
print("=" * 80)

print(results_df)

results_df.to_csv(

    "salmon_model_comparison_optimized.csv",

    index=False
)

print(
    "\nSaved: salmon_model_comparison_optimized.csv"
)



FINAL MODEL COMPARISON
                 Model  Accuracy  Precision  Recall      F1  Training_Time_s
9                  MLP    0.9506     0.9375  0.9056  0.9207            23.50
6              XGBoost    0.9410     0.9389  0.8754  0.8999            17.45
4          Extra Trees    0.9272     0.9082  0.8598  0.8728             2.65
8            SVM (RBF)    0.8921     0.8287  0.9455  0.8694             1.62
3        Random Forest    0.9264     0.9252  0.8393  0.8660             6.16
7                  KNN    0.9205     0.8986  0.8412  0.8631             0.01
2        Decision Tree    0.8674     0.8064  0.8359  0.8185             0.62
0  Logistic Regression    0.8389     0.7815  0.8944  0.8160             2.70
1          Gaussian NB    0.7075     0.6725  0.8073  0.6938             0.01
5             AdaBoost    0.8126     0.6992  0.6384  0.6557            71.99

Saved: salmon_model_comparison_optimized.csv
